In [1]:
import requests
import geopandas as gpd
from shapely.geometry import Polygon
import json
import warnings
from urllib3.exceptions import InsecureRequestWarning

c:\Users\ferdy\development\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
warnings.filterwarnings('ignore', category=InsecureRequestWarning)

In [3]:
def download_landuse_data_by_bbox():
    """
    Mengunduh data Penggunaan Tanah dari server BIG berdasarkan Bounding Box (BBOX)
    dan mem-parsing format Esri JSON.
    """
    # Endpoint query untuk layer Penggunaan Tanah 10K (Layer ID: 3)
    query_url = "https://kspservices.big.go.id/satupeta/rest/services/PUBLIK/SUMBER_DAYA_ALAM_DAN_LINGKUNGAN/MapServer/3/query"

    # Definisikan Bounding Box di sekitar D.I. Yogyakarta
    # Format: xmin, ymin, xmax, ymax dalam WGS84 (lat/lon)
    bbox_diy = {
        "xmin": 110.00,
        "ymin": -8.21,
        "xmax": 110.85,
        "ymax": -7.52
    }
    
    # Filter Geometri
    params = {
        'geometry': f"{bbox_diy['xmin']},{bbox_diy['ymin']},{bbox_diy['xmax']},{bbox_diy['ymax']}",
        'geometryType': 'esriGeometryEnvelope',
        'inSR': '4326', 
        'spatialRel': 'esriSpatialRelIntersects', 
        'outFields': '*',
        'outSR': '4326',
        'f': 'json',
        'resultOffset': 0,
        'resultRecordCount': 1000
    }
    
    print(f"Mulai mengunduh data Penggunaan Tanah untuk area di sekitar Yogyakarta...")
    print(f"Menghubungi server: {query_url}")
    
    all_features = []
    
    while True:
        try:
            print(f"Mengambil data dari offset: {params['resultOffset']}...")
            response = requests.get(query_url, params=params, timeout=90, verify=False)
            response.raise_for_status()
            data = response.json()
            
            if 'error' in data:
                print(f"Server memberikan error: {data['error']}")
                return None

            features = data.get('features', [])
            
            if features:
                all_features.extend(features)
                if 'exceededTransferLimit' in data and data['exceededTransferLimit'] is True:
                    params['resultOffset'] += len(features)
                else:
                    print("Pengambilan data dari server selesai.")
                    break
            else:
                print("Tidak ada lagi data yang ditemukan. Pengambilan selesai.")
                break

        except requests.exceptions.RequestException as e:
            print(f"Gagal mengambil data: {e}")
            return None
    
    if not all_features:
        print("Tidak ada fitur yang berhasil diunduh.")
        return None

    geometries = [Polygon(f['geometry']['rings'][0]) for f in all_features if f.get('geometry')]
    attributes = [f.get('attributes', {}) for f in all_features]
    
    if not geometries:
        print("Tidak ada data geometri yang valid ditemukan.")
        return None

    gdf = gpd.GeoDataFrame(attributes, geometry=geometries, crs="EPSG:4326")
    return gdf


In [4]:
gdf_penggunaan_lahan_diy = download_landuse_data_by_bbox()

Mulai mengunduh data Penggunaan Tanah untuk area di sekitar Yogyakarta...
Menghubungi server: https://kspservices.big.go.id/satupeta/rest/services/PUBLIK/SUMBER_DAYA_ALAM_DAN_LINGKUNGAN/MapServer/3/query
Mengambil data dari offset: 0...
Mengambil data dari offset: 1000...


KeyboardInterrupt: 

In [ ]:
if gdf_penggunaan_lahan_diy is not None and not gdf_penggunaan_lahan_diy.empty:
    output_filename = "Peta_Penggunaan_Lahan_DIY.geojson"
    try:
        gdf_penggunaan_lahan_diy.to_file(output_filename, driver='GeoJSON')
        print(f"\n[SUKSES] Data penggunaan lahan berhasil disimpan sebagai '{output_filename}'")
        print(f"Total poligon yang ditemukan di area Yogyakarta: {len(gdf_penggunaan_lahan_diy)}")
        print("\nKolom yang tersedia:", gdf_penggunaan_lahan_diy.columns.tolist())
        print("\nContoh jenis penggunaan lahan yang ditemukan:")
        print(gdf_penggunaan_lahan_diy['ptnobjname'].value_counts().head())
    except Exception as e:
        print(f"\n[ERROR] Gagal menyimpan file GeoJSON: {e}")
else:
    print("\nTidak ada data penggunaan lahan yang berhasil diproses.")